# Analisi Esplorativa dei Dati Clinici e delle Terapie (NLP)

Questo notebook presenta un'analisi approfondita del dataset preelaborato di anamnesi e terapie dei pazienti. Il dataset di partenza è stato strutturato estraendo informazioni cliniche testuali, analizzando le terapie all'ingresso ed alla dimissione ed eseguendo un allineamento (Medication Reconciliation) per tracciare farmaci aggiunti, sospesi o mantenuti.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Impostazione del layout per i grafici
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Definiamo i percorsi dei file
DATA_DIR = "../data"
STRUCTURED_JSON = os.path.join(DATA_DIR, "anamnesiterapie_structured.json")
STRUCTURED_CSV = os.path.join(DATA_DIR, "anamnesiterapie_structured.csv")

# Carichiamo il dataset
df = pd.read_csv(STRUCTURED_CSV)
with open(STRUCTURED_JSON, "r", encoding="utf-8") as f:
    patient_data = json.load(f)

print(f"Dataset caricato con successo! Numero totale di pazienti/ricoveri: {len(df)}")

## 1. Analisi delle Lunghezze dei Testi

Valutiamo la distribuzione della lunghezza dei testi clinici. Questa è un'informazione fondamentale per dimensionare i modelli di NLP (es. la dimensione del contesto nei modelli basati su Transformer come BERT o ClinicalBERT).

In [ ]:
# Calcoliamo le lunghezze in caratteri e in parole
df['len_anamnesi_char'] = df['anamnesi_testo'].fillna('').apply(len)
df['len_anamnesi_word'] = df['anamnesi_testo'].fillna('').apply(lambda x: len(x.split()))

df['len_ingresso_char'] = df['terapia_ingresso_testo'].fillna('').apply(len)
df['len_ingresso_word'] = df['terapia_ingresso_testo'].fillna('').apply(lambda x: len(x.split()))

df['len_dimissione_char'] = df['terapia_dimissione_testo'].fillna('').apply(len)
df['len_dimissione_word'] = df['terapia_dimissione_testo'].fillna('').apply(lambda x: len(x.split()))

# Visualizzazione delle statistiche descrittive
print("Statistiche delle lunghezze in parole:")
print(df[['len_anamnesi_word', 'len_ingresso_word', 'len_dimissione_word']].describe().round(1))

In [ ]:
# Distribuzione delle lunghezze delle anamnesi
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['len_anamnesi_word'], bins=30, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Distribuzione Parole nelle Anamnesi')
axes[0].set_xlabel('Numero di parole')
axes[0].set_ylabel('Frequenza')

# Distribuzione delle lunghezze delle terapie
sns.kdeplot(df['len_ingresso_word'], label="Terapia Ingresso", fill=True, alpha=0.3, ax=axes[1], color='coral')
sns.kdeplot(df['len_dimissione_word'], label="Terapia Dimissione", fill=True, alpha=0.3, ax=axes[1], color='royalblue')
axes[1].set_title('Confronto Lunghezza Parole Terapie')
axes[1].set_xlabel('Numero di parole')
axes[1].set_ylabel('Densità')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Prevalenza delle Comorbilità

Abbiamo estratto tramite regole basate su parole chiave e regex le principali comorbilità e fattori di rischio dall'anamnesi clinica. Questo tipo di feature extraction è utile sia per la profilazione dei pazienti che come etichette per task di classificazione multilabel.

In [ ]:
comorbidities = [
    "ipertensione", "diabete", "dislipidemia", "obesita", 
    "fibrillazione_atriale", "cardiopatia_ischemica", 
    "scompenso_cardiaco", "insufficienza_renale", "distiroidismo", "fumo"
]

# Calcoliamo la somma di pazienti positivi per comorbilità
counts = df[comorbidities].sum().sort_values(ascending=False)
percentages = (counts / len(df) * 100).round(1)

# Creiamo un dataframe per il grafico
df_comorb = pd.DataFrame({'Conteggio': counts, 'Percentuale': percentages})
print(df_comorb)

# Grafico a barre
sns.barplot(x=df_comorb.index, y=df_comorb['Conteggio'], palette="viridis")
plt.title('Prevalenza delle Comorbilità e Fattori di Rischio nelle Anamnesi')
plt.ylabel('Numero di pazienti')
plt.xlabel('Condizione clinica')
plt.xticks(rotation=45)

# Aggiungiamo le etichette delle percentuali sopra le barre
for i, p in enumerate(percentages):
    plt.text(i, counts[i] + 15, f"{p}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Riconciliazione Terapeutica (Medication Reconciliation)

La riconciliazione terapeutica è un task critico in ambito clinico per identificare le variazioni della terapia farmacologica tra l'ammissione e la dimissione del paziente. 
Valutiamo quanti farmaci vengono mantenuti, sospesi o aggiunti durante la degenza.

In [ ]:
# Estraiamo i conteggi di farmaci per paziente
df['num_ingresso'] = df['terapia_ingresso_farmaci'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)
df['num_dimissione'] = df['terapia_dimissione_farmaci'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)
df['num_mantenuti'] = df['farmaci_mantenuti'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)
df['num_sospesi'] = df['farmaci_sospesi'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)
df['num_aggiunti'] = df['farmaci_aggiunti'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)

print("Media dei farmaci gestiti per ricovero:")
print(df[['num_ingresso', 'num_dimissione', 'num_mantenuti', 'num_sospesi', 'num_aggiunti']].mean().round(2))

In [ ]:
# Visualizziamo la distribuzione del numero di farmaci per paziente tra ingresso e dimissione
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df[['num_ingresso', 'num_dimissione']], palette="pastel", ax=axes[0])
axes[0].set_title('Numero di Farmaci: Ingresso vs Dimissione')
axes[0].set_ylabel('Numero di farmaci')
axes[0].set_xticklabels(['All\'ingresso', 'Alla dimissione'])

# Analizziamo i cambiamenti terapeutici
med_rec_means = df[['num_mantenuti', 'num_sospesi', 'num_aggiunti']].mean()
sns.barplot(x=med_rec_means.index, y=med_rec_means.values, palette="muted", ax=axes[1])
axes[1].set_title('Media dei Cambiamenti Terapeutici per Paziente')
axes[1].set_ylabel('Numero medio di farmaci')
axes[1].set_xticklabels(['Mantenuti', 'Sospesi (Stop)', 'Aggiunti (Start)'])

plt.tight_layout()
plt.show()

## 4. Analisi dei Farmaci più Comuni

Scopriamo quali sono i farmaci e i principi attivi maggiormente prescritti all'ingresso e alla dimissione nel dataset di 1000 pazienti.

In [ ]:
# Estrazione di tutti i farmaci all'ingresso
ingresso_drugs = []
for patient in patient_data:
    for drug in patient.get('terapia_ingresso_farmaci', []):
        # prendiamo solo la prima parola o il nome pulito
        ingresso_drugs.append(drug.get('nome').split()[0].title())

# Estrazione di tutti i principi attivi alla dimissione
dimissione_drugs = []
for patient in patient_data:
    for drug in patient.get('terapia_dimissione_farmaci', []):
        principio = drug.get('principio_attivo').split('/')[0].split('(')[0].strip().title()
        if principio and principio != "Nessuna":
            dimissione_drugs.append(principio)

common_ingresso = Counter(ingresso_drugs).most_common(15)
common_dimissione = Counter(dimissione_drugs).most_common(15)

# Creiamo il grafico per i farmaci più comuni all'ingresso
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.barplot(
    x=[count for _, count in common_ingresso], 
    y=[drug for drug, _ in common_ingresso], 
    ax=axes[0], 
    palette="Reds_r"
)
axes[0].set_title('Top 15 Farmaci/Brand all\'Ingresso')
axes[0].set_xlabel('Numero di prescrizioni')

# Creiamo il grafico per i principi attivi più comuni alla dimissione
sns.barplot(
    x=[count for _, count in common_dimissione], 
    y=[drug for drug, _ in common_dimissione], 
    ax=axes[1], 
    palette="Blues_r"
)
axes[1].set_title('Top 15 Principi Attivi alla Dimissione')
axes[1].set_xlabel('Numero di prescrizioni')

plt.tight_layout()
plt.show()

## 5. Prossimi Passi per Task di NLP

Questo dataset strutturato offre eccellenti basi per diversi task avanzati di NLP in ambito clinico:

1. **Clinical Named Entity Recognition (NER)**: 
   * Addestramento di modelli (es. Spacy o Hugging Face Transformers) per estrarre entità come *Patologie*, *Sintomi*, *Farmaci*, *Dosaggi* direttamente dall'anamnesi.
   * È possibile utilizzare il dataset strutturato per generare un dataset etichettato in formato BIO/CoNLL.

2. **Classificazione Multi-label del Profilo Paziente**:
   * Addestrare un classificatore di testo (es. BERT o LSTM) che riceve in input l'anamnesi testuale ed predice le comorbilità del paziente (classi binarie estratte in questa fase).

3. **Medication Reconciliation Predittiva**:
   * Modellare la probabilità che un farmaco assunto all'ingresso venga sospeso o modificato in base al quadro clinico descritto nell'anamnesi.

4. **Information Extraction / Relation Extraction**:
   * Associare ciascun farmaco alle istruzioni di dosaggio (`istruzioni`) ed estrarre relazioni tipo: `(Farmaco) -> somministrato_a -> (Dosaggio)` o `(Farmaco) -> indicato_per -> (Patologia)`.